# Navigator Ascon demo preparation
Derived from the existing jupyter_demo.ipynb host flow; the original notebook is unchanged.
Default: **simulated** Mailbox Transport, no measured cycles and no FPGA programming.
Real mode requires a reviewed physical driver, matching bit/hwh/firmware manifest and HWH bindings.
The shipped profile intentionally reports **NOT_CONFIGURED**. Public inputs below are test-only.


In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/pynq/ascon_host_api.py').is_file())
sys.path.insert(0, str(root / 'src/pynq'))
from ascon_host_api import AsconHost, AuthenticationError
from mailbox_backend import MailboxBackend, MemoryTransport
from navigator_backend import NavigatorTransport, NotConfiguredError, create_host
from golden_checker import parse_kat
USE_HARDWARE = False
# Future: caller-loaded overlay and physical Transport driver. No download here.
physical_driver = None
loaded_overlay = None
profile_path = root / 'board/atk_navigator_7020/backend_profile.json'


In [ ]:
# User inputs (hex); limits come from backend capabilities, not this UI.
key_hex = '000102030405060708090a0b0c0d0e0f'
nonce_hex = '101112131415161718191a1b1c1d1e1f'
ad_hex = ''
plaintext_hex = '000102030405060708090a0b0c0d0e0f'
key, nonce, ad, plaintext = map(bytes.fromhex, (key_hex, nonce_hex, ad_hex, plaintext_hex))
results = {}
hosts = {}
for mode in ('sw', 'hw', 'ise'):
    try:
        if USE_HARDWARE:
            host = create_host(profile_path, mode, driver=physical_driver, overlay=loaded_overlay)
        else:
            host = AsconHost(MailboxBackend(NavigatorTransport({'status': 'mock'}, driver=MemoryTransport()), mode))
        result = host.encrypt(key, nonce, ad, plaintext)
        recovered = host.decrypt(key, nonce, ad, result.ciphertext, result.tag)
        assert recovered.plaintext == plaintext
        hosts[mode], results[mode] = host, result
        print(mode, 'simulated' if result.simulated else 'hardware', 'CT=', result.ciphertext.hex(),
              'tag=', result.tag.hex(), 'cycles=', result.cycles, 'cycle_status=', result.cycle_info.status)
    except NotConfiguredError:
        print(mode, 'NOT_CONFIGURED: physical mailbox/driver/overlay pending')


In [ ]:
# Independent official expected ciphertext/tag, not just encrypt/decrypt round trip.
cases = parse_kat()
for mode, host in hosts.items():
    for case in cases[:2]:
        k, n, a, p = (case[field] for field in ('Key', 'Nonce', 'AD', 'PT'))
        output = host.encrypt(k, n, a, p)
        assert output.ciphertext + output.tag == case['CT']
    result = results[mode]
    try:
        host.decrypt(key, nonce, ad, result.ciphertext, bytes([result.tag[0] ^ 1]) + result.tag[1:])
    except AuthenticationError:
        print(mode, 'official sample KAT PASS; tampered tag rejected')
    else:
        raise AssertionError('tampered tag accepted')
# Full official KAT: python src/pynq/golden_checker.py --backend mailbox --mode ise


In [ ]:
# Do not mix historical simulation samples with current FPGA measurements.
for mode, result in results.items():
    sw = results.get('sw')
    comparable = (sw is not None and not sw.simulated and not result.simulated
                  and sw.cycles is not None and result.cycles is not None and result.cycles > 0
                  and sw.cycle_info.status == result.cycle_info.status == 'hardware_measured'
                  and sw.cycle_info.scope == result.cycle_info.scope)
    speedup = sw.cycles / result.cycles if comparable else None
    print(mode, 'speedup=', speedup if speedup is not None else 'NOT_MEASURED')


Hardware handoff: use the future driver behind NavigatorTransport, keeping AsconHost and MailboxBackend.
PS/PL mapping, cache ownership, reset/boot and recovery still require integration and real-board validation.
No physical addresses or control offsets are assigned by this notebook.
